# 04 CT Experiments (v2: Full-Data Self-Supervised Pretraining + LoRA/Full Fine-Tuning + Multi-Seed)

This version focuses on a paper-ready CT experiment workflow:

- **Pretraining (self-supervised)**: contrastive learning over the full `df` (labels are optional)
- **Fine-tuning / evaluation (supervised)**: time-series split and evaluation on `df_labeled` for target countries
- **Recommended main methods**: `C+CT (full)` and `D_lora+CT`
- **Reporting**: mean±std over multiple random seeds (no cherry-picking of the best test run)

By default, heavy runs are disabled; start with a small subset of countries for a smoke test.

In [1]:
# 1) Environment and paths
import os, sys, time, pickle
import numpy as np
import pandas as pd

project_root = os.path.abspath(os.getcwd())
if not os.path.exists(os.path.join(project_root, 'data')):
    candidate = os.path.abspath(os.path.join(project_root, 'Transfer learning'))
    if os.path.exists(os.path.join(candidate, 'data')):
        project_root = candidate
    else:
        project_root = '/hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning'

data_dir = os.path.join(project_root, 'data')
results_dir = os.path.join(project_root, 'results')
os.makedirs(results_dir, exist_ok=True)

module_path = os.path.abspath(os.path.join(project_root, 'python modules'))
if module_path not in sys.path:
    sys.path.append(module_path)

print('✅ project_root =', project_root)
print('✅ results_dir  =', results_dir)

✅ project_root = /hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning
✅ results_dir  = /hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/results


In [2]:
# 2) Data loading: keep df (full) + df_labeled (all 6 targets available)
processed_hdf5_path = os.path.join(data_dir, 'processed_feature_data.h5')
TARGET_COLS = [
    'log_Scope1', 'log_Scope2', 'log_Scope3_upstream',
    'log_Scope3_downLA', 'log_Scope3_prod', 'log_Scope_total'
]

possible_keys = ['processed_data', 'processed_feature_data', 'data', 'df_processed', '/processed_features']
df = None
with pd.HDFStore(processed_hdf5_path, 'r') as store:
    available_keys = list(store.keys())
    for key in possible_keys:
        if key in available_keys:
            df = pd.read_hdf(processed_hdf5_path, key=key)
            print('✅ HDF5 key =', key)
            break
    if df is None:
        df = pd.read_hdf(processed_hdf5_path, key=available_keys[0])
        print('✅ HDF5 key =', available_keys[0])

# Clean loc
valid_loc = (
    df['loc'].notna() & (df['loc'] != '') & (df['loc'].astype(str).str.strip() != '')
    & (df['loc'].astype(str).str.len() >= 2)
    & (df['loc'].astype(str).str.isalpha())
    & (df['loc'].astype(str).str.len() <= 10)
)
df = df[valid_loc].copy()

# Feature columns
id_cols = ['gvkey', 'fiscalyear', 'loc', 'GICSSector']
exclude_cols = list(TARGET_COLS) + id_cols
original_targets = ['Scope1', 'Scope2', 'Scope3_upstream', 'Scope3_prod', 'Scope3_downLA', 'Scope_total']
exclude_cols.extend([c for c in original_targets if c in df.columns])
other_exclude = [c for c in df.columns if 'sector' in c.lower() and c not in id_cols]
exclude_cols.extend(other_exclude)
FEATURE_COLS = [c for c in df.columns if c not in exclude_cols]

df_labeled = df[df[TARGET_COLS].notnull().all(axis=1)].copy()

country_counts = df_labeled['loc'].value_counts().sort_values(ascending=False)
eligible_countries = country_counts[country_counts >= 50].index.tolist()

print('✅ df shape       =', df.shape)
print('✅ df_labeled     =', df_labeled.shape)
print('✅ #features      =', len(FEATURE_COLS))
print('✅ #eligible>=50  =', len(eligible_countries))

✅ HDF5 key = /processed_features
✅ df shape       = (1473470, 110)
✅ df_labeled     = (85223, 110)
✅ #features      = 100
✅ #eligible>=50  = 55


In [ ]:
# 3) Import CT module (supports df_pretrain/df_finetune + finetune_strategy='lora'/'full')
import os

# Use 4 GPUs for this notebook
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0,1,2,3')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)

from ct_cctl_emit import run_ct_cctl_emit_experiment, run_automl_experiment, run_ct_cctl_pretrain_only

# QUICK_RUN mode: enable smaller models and fewer epochs for fast completion
QUICK_RUN = False

# ========= VERSIONING =========
CONFIG_VERSION = 'v3.4'
RUN_TAG = f'ct_{CONFIG_VERSION}'

# ========== v3.4 CONFIGURATION (shared pretrain) ==========
# NOTE: Do NOT change d_model (reuse existing shared pretrain)

CT_BASE_CFG = {
    # ========== Model architecture ==========
    'd_model': 384,   # keep fixed for pretrain reuse
    'n_heads': 12,
    'n_layers': 4,
    
    # ========== Training epochs ==========
    'pretrain_epochs': 60,
    'finetune_epochs': 600,
    
    # ========== Training hyperparameters ==========
    'temperature': 0.07,
    'pretrain_lr': 3e-4,
    'finetune_lr': 2e-5,         # base; strategy overrides below
    'finetune_patience': 60,
    'finetune_weight_decay': 1e-4,
    
    # ========== Batch sizes ==========
    'pretrain_batch_size': 2048,
    'finetune_batch_size': 128,
    
    # ========== DataLoader settings ==========
    'loader_num_workers': 16,
    'loader_pin_memory': (device == 'cuda'),
    'loader_persistent_workers': True,
    'loader_prefetch_factor': 4,

    # ========== AMP (mixed precision) ==========
    'use_amp': True,
    
    # ========== LoRA hyperparameters ==========
    'lora_rank': 32,
    'lora_alpha': 64,

    # ========== CT + MLP Stacking (baseline fusion) ==========
    'ensemble_strategy': 'ct_mlp_stack',
    'mlp_hidden_sizes': (1024, 512),
    'mlp_dropout': 0.1,
    'mlp_epochs': 500,
    'mlp_lr': 3e-4,
    'mlp_batch_size': 256,
    'mlp_patience': 50,
    'mlp_weight_decay': 5e-5,

    'device': device,
    'config_version': CONFIG_VERSION,
    'run_tag': RUN_TAG,
}

# ========== Strategy-specific configs ==========
# C_CT = FULL fine-tune (freeze_encoder=False)
# D_lora_CT = LoRA fine-tune (freeze base, train LoRA + head)

CT_FULL_CFG = dict(CT_BASE_CFG)
CT_FULL_CFG['freeze_encoder'] = False
CT_FULL_CFG['finetune_lr'] = 1e-5      # lower LR for full fine-tune

CT_LORA_CFG = dict(CT_BASE_CFG)
CT_LORA_CFG['freeze_encoder'] = True
CT_LORA_CFG['finetune_lr'] = 2e-5

# QUICK_RUN overrides: much smaller model and fewer epochs
def _apply_quick_overrides(cfg: dict, lora: bool = False):
    cfg['d_model'] = 128
    cfg['n_heads'] = 4
    cfg['n_layers'] = 2
    cfg['pretrain_epochs'] = 3
    cfg['finetune_epochs'] = 10
    cfg['pretrain_batch_size'] = 4096
    cfg['finetune_batch_size'] = 512
    cfg['loader_num_workers'] = max(2, (os.cpu_count() or 8) // 4)
    cfg['loader_prefetch_factor'] = 2
    if lora:
        cfg['lora_rank'] = 8
        cfg['lora_alpha'] = 16

if QUICK_RUN:
    print('🔧 QUICK_RUN enabled for CT experiments: reducing model size and epochs')
    _apply_quick_overrides(CT_FULL_CFG, lora=False)
    _apply_quick_overrides(CT_LORA_CFG, lora=True)

# Run only full strategy (C_CT)
RUN_STRATEGIES = ['full']  # 'full' = C+CT
STRATEGY_LABELS = {'full': 'C_CT'}

print('CONFIG_VERSION =', CONFIG_VERSION)
print('CT_FULL_CFG =')
for k, v in CT_FULL_CFG.items():
    print(f'   {k}: {v}')
print()
print('CT_LORA_CFG =')
for k, v in CT_LORA_CFG.items():
    print(f'   {k}: {v}')

print(
    f"""
========== v3.4 Configuration {CONFIG_VERSION} ==========
   d_model fixed (reuse shared pretrain)
   Pretrain: epochs=60
   Finetune: epochs=600, patience=60
   Full FT LR: 1e-5
   MLP: (1024,512), epochs=500, lr=3e-4
================================================
"""
 )

device = cuda
CONFIG_VERSION = v3.5_max
CT_FULL_CFG =
   d_model: 384
   n_heads: 12
   n_layers: 4
   pretrain_epochs: 60
   finetune_epochs: 1200
   temperature: 0.07
   pretrain_lr: 0.0003
   finetune_lr: 1e-05
   finetune_patience: 120
   finetune_weight_decay: 5e-05
   pretrain_batch_size: 2048
   finetune_batch_size: 128
   loader_num_workers: 16
   loader_pin_memory: True
   loader_persistent_workers: True
   loader_prefetch_factor: 4
   use_amp: True
   lora_rank: 64
   lora_alpha: 128
   ensemble_strategy: ct_mlp_stack
   mlp_hidden_sizes: (2048, 1024, 512)
   mlp_dropout: 0.08
   mlp_epochs: 800
   mlp_lr: 0.0002
   mlp_batch_size: 256
   mlp_patience: 80
   mlp_weight_decay: 3e-05
   device: cuda
   config_version: v3.5_max
   run_tag: ct_v3.5_max
   freeze_encoder: False

CT_LORA_CFG =
   d_model: 384
   n_heads: 12
   n_layers: 4
   pretrain_epochs: 60
   finetune_epochs: 1200
   temperature: 0.07
   pretrain_lr: 0.0003
   finetune_lr: 2e-05
   finetune_patience: 120
   fi

In [4]:
# 3.5) Optional: Optuna AutoML (small subset, then apply best params)
RUN_OPTUNA = False  # set True to run
OPTUNA_COUNTRIES = eligible_countries[:2]  # small subset for tuning
OPTUNA_TRIALS = 20
OPTUNA_TIMEOUT = 60 * 60  # seconds
OPTUNA_SEED = 42

# Optuna checkpoint (resume)
OPTUNA_DIR = os.path.join(results_dir, 'checkpoints', 'ct_optuna')
os.makedirs(OPTUNA_DIR, exist_ok=True)

if RUN_OPTUNA:
    # Use the first country in the subset as the tuning target
    target_country = OPTUNA_COUNTRIES[0]
    print(f'🔎 Optuna tuning target: {target_country}')

    study_name = f'ct_cctl_emit_{target_country}'
    storage = f"sqlite:///{os.path.join(OPTUNA_DIR, study_name + '.db')}"
    print(f'📌 Optuna storage: {storage}')

    automl_result = run_automl_experiment(
        df=df,
        features=FEATURE_COLS,
        targets=TARGET_COLS,
        target_country=target_country,
        n_trials=OPTUNA_TRIALS,
        timeout=OPTUNA_TIMEOUT,
        device=device,
        random_state=OPTUNA_SEED,
    )

    best_params = automl_result.get('best_params', {})
    print('Best params from Optuna:', best_params)

    # Apply Optuna best params to CT_BASE_CFG (safe keys only)
    for k in ['d_model','n_heads','n_layers','temperature','pretrain_epochs','finetune_epochs']:
        if k in best_params:
            CT_BASE_CFG[k] = best_params[k]
    if 'pretrain_lr' in best_params:
        CT_BASE_CFG['pretrain_lr'] = best_params['pretrain_lr']
    if 'finetune_lr' in best_params:
        CT_BASE_CFG['finetune_lr'] = best_params['finetune_lr']
    if 'finetune_batch_size' in best_params:
        CT_BASE_CFG['finetune_batch_size'] = best_params['finetune_batch_size']
    if 'freeze_encoder' in best_params:
        CT_BASE_CFG['freeze_encoder'] = best_params['freeze_encoder']

    # Refresh strategy configs after update
    CT_FULL_CFG = dict(CT_BASE_CFG)
    CT_LORA_CFG = dict(CT_BASE_CFG)
    print('✅ CT_BASE_CFG updated from Optuna')

In [5]:
# 3.7) Shared pretraining (run once; reuse encoder for all countries)
# NOTE: Set RUN_SHARED_PRETRAIN=True to re-run pretraining with new config

FORCE_RERUN = False  # Set to True to clear old checkpoints and re-run everything

RUN_SHARED_PRETRAIN = FORCE_RERUN  # Re-run pretraining with improved config
PRETRAIN_DIR = os.path.join(results_dir, 'checkpoints', 'ct_shared_pretrain_v3')  # New dir for v3
PRETRAIN_CKPT_PATH = os.path.join(PRETRAIN_DIR, 'pretrain_checkpoint.pt')

# Clear old v2 checkpoints if forcing rerun
if FORCE_RERUN:
    import shutil
    old_dirs = [
        os.path.join(results_dir, 'checkpoints', 'c_ct_v2'),
        os.path.join(results_dir, 'checkpoints', 'd_lora_ct_v2'),
        os.path.join(results_dir, 'checkpoints', 'ct_shared_pretrain'),
    ]
    for d in old_dirs:
        if os.path.exists(d):
            print(f'🗑️ Removing old checkpoint dir: {d}')
            shutil.rmtree(d)

# Ensure pretraining uses the configured batch size
PRETRAIN_BATCH_SIZE = CT_BASE_CFG['pretrain_batch_size']
print(f'🚀 Pretrain batch size: {PRETRAIN_BATCH_SIZE}')

if RUN_SHARED_PRETRAIN or not os.path.exists(PRETRAIN_CKPT_PATH):
    print('🚀 Running shared pretraining (v3 improved with 4-GPU DataParallel) ...')
    run_ct_cctl_pretrain_only(
        df=df,
        df_pretrain=df,
        features=FEATURE_COLS,
        pretrain_epochs=CT_BASE_CFG['pretrain_epochs'],
        pretrain_lr=CT_BASE_CFG['pretrain_lr'],
        pretrain_batch_size=PRETRAIN_BATCH_SIZE,
        d_model=CT_BASE_CFG['d_model'],
        n_heads=CT_BASE_CFG['n_heads'],
        n_layers=CT_BASE_CFG['n_layers'],
        n_targets=len(TARGET_COLS),
        temperature=CT_BASE_CFG['temperature'],
        device=CT_BASE_CFG['device'],
        checkpoint_dir=PRETRAIN_DIR,
        dataloader_num_workers=CT_BASE_CFG['loader_num_workers'],
        dataloader_pin_memory=CT_BASE_CFG['loader_pin_memory'],
        dataloader_persistent_workers=CT_BASE_CFG['loader_persistent_workers'],
        dataloader_prefetch_factor=CT_BASE_CFG['loader_prefetch_factor'],
        use_amp=CT_BASE_CFG.get('use_amp', False),
        verbose=True,
    )
else:
    print('✅ Found shared pretrain checkpoint:', PRETRAIN_CKPT_PATH)

# Reuse shared encoder for all downstream fine-tuning
CT_FULL_CFG['pretrain_checkpoint_path'] = PRETRAIN_CKPT_PATH
CT_LORA_CFG['pretrain_checkpoint_path'] = PRETRAIN_CKPT_PATH
CT_FULL_CFG['skip_pretrain'] = True
CT_LORA_CFG['skip_pretrain'] = True

🚀 Pretrain batch size: 2048
✅ Found shared pretrain checkpoint: /hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/results/checkpoints/ct_shared_pretrain_v3/pretrain_checkpoint.pt


In [ ]:
# 4) CT main experiment: full-data pre-training + (C+CT full) fine-tuning (4 GPUs)
from tqdm.auto import tqdm
import json
import multiprocessing as mp
import shutil
import glob

from ct_cctl_worker import ct_worker_run

RUN_MAIN = True  # set to True to start running
CLEAR_CT_MAIN_CHECKPOINTS = False  # True = clear CT main checkpoints and re-run

# Save a config snapshot before running (for crash safety)
pre_cfg_path = os.path.join(results_dir, f'ct_config_{RUN_TAG}_pre.json')
with open(pre_cfg_path, 'w') as f:
    json.dump({'CT_BASE_CFG': CT_BASE_CFG, 'CT_FULL_CFG': CT_FULL_CFG, 'CT_LORA_CFG': CT_LORA_CFG}, f, indent=2)
print('Saved pre-run config:', pre_cfg_path)

# Clear ONLY CT main checkpoints (keep shared pretrain)
if CLEAR_CT_MAIN_CHECKPOINTS:
    ct_ckpt_dirs = [
        os.path.join(results_dir, 'checkpoints', 'c_ct_v3'),
        os.path.join(results_dir, 'checkpoints', 'd_lora_ct_v3'),
    ]
    for d in ct_ckpt_dirs:
        if os.path.exists(d):
            print(f'🗑️ Removing CT main checkpoint dir: {d}')
            shutil.rmtree(d)

# ========== FULL COUNTRY LIST (>=50 samples) ==========
country_counts = df_labeled['loc'].value_counts().sort_values(ascending=False)
eligible_sorted = [c for c in country_counts.index if c in eligible_countries]

# One seed only
SEEDS = [42]

# Load previous v3.4 results to skip already-run countries (same seed)
def _latest_run_csv(tag):
    paths = glob.glob(os.path.join(results_dir, f'ct_experiments_{tag}_*.csv'))
    if not paths:
        return None
    paths.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    return paths[0]

prev_csv = _latest_run_csv(RUN_TAG)
completed_countries = set()
prev_df = None
if prev_csv and os.path.exists(prev_csv):
    prev_df = pd.read_csv(prev_csv)
    if 'experiment_code' not in prev_df.columns:
        prev_df['experiment_code'] = prev_df.get('experiment_type', prev_df.get('experiment', ''))
    completed_countries = set(
        prev_df[(prev_df['experiment_code'] == 'C_CT') & (prev_df['seed'].isin(SEEDS))]['country'].unique()
    )
    print(f"Found previous run: {prev_csv}")
    print(f"Skip {len(completed_countries)} already-run countries for seed {SEEDS}")
else:
    print("No previous run found; will run all eligible countries")

COUNTRIES = [c for c in eligible_sorted if c not in completed_countries]
print(f"Total eligible countries: {len(eligible_sorted)}")
print(f"Countries to run now: {len(COUNTRIES)}")

# Multi-GPU settings
NUM_WORKERS = 4
GPU_IDS = [0, 1, 2, 3]

def _balanced_split_countries(df_labeled_df, countries_list, n):
    """Split countries into n shards balanced by sample counts."""
    counts = df_labeled_df['loc'].value_counts()
    ordered = [(c, counts.get(c, 0)) for c in countries_list if c in counts.index]
    ordered.sort(key=lambda x: -x[1])

    shards = [[] for _ in range(n)]
    loads = [0] * n

    for country, cnt in ordered:
        min_idx = loads.index(min(loads))
        shards[min_idx].append(country)
        loads[min_idx] += cnt

    return shards, loads

rows = []
if RUN_MAIN:
    total_runs = len(SEEDS) * len(COUNTRIES) * len(RUN_STRATEGIES)
    print(f'\n🚀 Starting CT experiments ({RUN_TAG}): {len(COUNTRIES)} countries × {len(SEEDS)} seeds × {len(RUN_STRATEGIES)} strategies = {total_runs} runs')
    
    shards, loads = _balanced_split_countries(df_labeled, COUNTRIES, NUM_WORKERS)
    print('Country assignment:')
    for i, (s, l) in enumerate(zip(shards, loads)):
        print(f'   Worker {i}: {len(s)} countries, ~{l} samples')

    ctx = mp.get_context('spawn')
    result_q = ctx.Queue()
    procs = []

    for i in range(NUM_WORKERS):
        if not shards[i]:
            continue
        p = ctx.Process(
            target=ct_worker_run,
            args=(
                result_q,
                i,
                GPU_IDS[i],
                project_root,
                results_dir,
                shards[i],
                SEEDS,
                RUN_STRATEGIES,
                STRATEGY_LABELS,
                CT_FULL_CFG,
                CT_LORA_CFG,
            ),
        )
        p.start()
        procs.append(p)

    # Progress bar
    pbar = tqdm(total=total_runs, desc=f'Total progress (CT {RUN_TAG})', leave=True)
    done_workers = 0
    worker_pkls = []

    while done_workers < len(procs):
        msg = result_q.get()
        if msg.get('type') == 'progress':
            pbar.update(msg.get('count', 1))
        elif msg.get('type') == 'done':
            done_workers += 1
            worker_pkls.append(msg.get('pkl'))
            print(f"✅ Worker {msg.get('worker_id')} done. Results: {msg.get('pkl')}")

    pbar.close()

    # Join workers
    for p in procs:
        p.join()

    # Aggregate new results
    all_rows = []
    for pkl_path in worker_pkls:
        if pkl_path and os.path.exists(pkl_path):
            with open(pkl_path, 'rb') as f:
                all_rows.extend(pickle.load(f))

    if all_rows or prev_df is not None:
        df_new = pd.DataFrame(all_rows) if all_rows else pd.DataFrame([])
        ts = time.strftime('%Y%m%d_%H%M%S')
        
        # Combine with previous results (append)
        if prev_df is not None and not prev_df.empty:
            df_all = pd.concat([prev_df, df_new], ignore_index=True)
        else:
            df_all = df_new
        
        out_csv = os.path.join(results_dir, f'ct_experiments_{RUN_TAG}_combined_{ts}.csv')
        out_pkl = os.path.join(results_dir, f'ct_experiments_{RUN_TAG}_combined_{ts}.pkl')
        out_cfg = os.path.join(results_dir, f'ct_config_{RUN_TAG}_{ts}.json')
        df_all.to_csv(out_csv, index=False)
        with open(out_pkl, 'wb') as f:
            pickle.dump(df_all.to_dict('records'), f)
        with open(out_cfg, 'w') as f:
            json.dump({'CT_BASE_CFG': CT_BASE_CFG, 'CT_FULL_CFG': CT_FULL_CFG, 'CT_LORA_CFG': CT_LORA_CFG}, f, indent=2)
        print('Saved combined raw:', out_csv)
        print('Saved config:', out_cfg)

        # ========== Paper-ready reporting: mean±std over seeds ==========
        print('\n' + '='*70)
        print('PAPER-READY RESULTS (mean±std over seeds, full country set)')
        print('='*70)
        
        if 'experiment_code' not in df_all.columns:
            df_all['experiment_code'] = df_all.get('experiment_type', df_all.get('experiment', ''))
        per_seed = df_all.groupby(['seed', 'experiment_code'])['R2_Average'].mean().reset_index()
        print('\nMean test R² per seed (avg over countries):')
        print(per_seed.to_string(index=False))
        
        summary = per_seed.groupby('experiment_code')['R2_Average'].agg(['mean', 'std', 'count']).reset_index()
        summary['report'] = summary.apply(lambda r: f"{r['mean']*100:.2f}% ± {r['std']*100:.2f}%", axis=1)
        print('\nOverall mean±std over seeds (by experiment):')
        print(summary[['experiment_code', 'report', 'count']].to_string(index=False))
        print('='*70)
    else:
        print('No new rows and no previous results found; nothing to save.')
else:
    print('RUN_MAIN=False (not running). Set to True when ready.')

Saved pre-run config: /hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/results/ct_config_ct_v3.5_max_pre.json
========== STRATIFIED SAMPLING (9 Countries) ==========
High sample count (18 total):
   - USA: 15514 samples
   - HKG: 2678 samples
   - DEU: 1349 samples
Medium sample count (18 total):
   - SGP: 817 samples
   - NOR: 486 samples
   - FIN: 346 samples
Low sample count (19 total):
   - RUS: 257 samples
   - KWT: 191 samples
   - NGA: 91 samples
Total selected: 9 countries

🚀 Starting CT experiments (ct_v3.5_max): 9 countries × 3 seeds × 1 strategies = 27 runs
Country assignment:
   Worker 0: 1 countries, ~15514 samples
   Worker 1: 1 countries, ~2678 samples
   Worker 2: 3 countries, ~1797 samples
   Worker 3: 4 countries, ~1740 samples


Total progress (CT ct_v3.5_max):   0%|          | 0/27 [00:00<?, ?it/s]

✅ Worker 3 done. Results: /hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/results/ct_worker_3.pkl
✅ Worker 0 done. Results: /hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/results/ct_worker_0.pkl
✅ Worker 2 done. Results: /hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/results/ct_worker_2.pkl
✅ Worker 1 done. Results: /hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/results/ct_worker_1.pkl
Saved raw: /hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/results/ct_experiments_ct_v3.5_max_20260207_093208.csv
Saved config: /hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/results/ct_config_ct_v3.5_max_20260207_093208.json

PAPER-READY RESULTS (mean±std over 3 seeds, stratified 9-country sample)

Mean test R² per seed (avg over 9 countries):
 seed experiment_code  R2_

In [7]:
# 5) Compare current vs previous CT configs
import json
from pathlib import Path

def _load_json(path):
    with open(path, 'r') as f:
        return json.load(f)

def _latest_config_for_tag(results_dir_path, run_tag):
    candidates = list(Path(results_dir_path).glob(f"ct_config_{run_tag}_*.json"))
    if not candidates:
        return None
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]

# Pick current and previous config files (auto-select latest for current run tag)
current_cfg_path = _latest_config_for_tag(results_dir, RUN_TAG)
previous_cfg_path = Path(results_dir) / "ct_config_backfilled_ct_experiments_stratified_20260205_172919.json"

print("Current:", current_cfg_path)
print("Previous:", previous_cfg_path)

if current_cfg_path is None:
    raise FileNotFoundError(f"No ct_config_{{RUN_TAG}}_*.json found under {results_dir}")

current_cfg = _load_json(current_cfg_path)
previous_cfg = _load_json(previous_cfg_path)

# Flatten key sections for comparison
current_base = current_cfg.get("CT_BASE_CFG", {})
current_full = current_cfg.get("CT_FULL_CFG", {})
current_lora = current_cfg.get("CT_LORA_CFG", {})

previous_configs = previous_cfg.get("configs", {})
previous_c = previous_configs.get("C_CT", {})
previous_d = previous_configs.get("D_lora_CT", {})

def _diff_dict(a, b):
    keys = sorted(set(a) | set(b))
    diffs = []
    for k in keys:
        va = a.get(k, "<missing>")
        vb = b.get(k, "<missing>")
        if va != vb:
            diffs.append((k, va, vb))
    return diffs

print("\n=== C_CT vs current CT_FULL_CFG ===")
for k, v_old, v_new in _diff_dict(previous_c, current_full):
    print(f"{k}: prev={v_old} | curr={v_new}")

print("\n=== D_lora_CT vs current CT_LORA_CFG ===")
for k, v_old, v_new in _diff_dict(previous_d, current_lora):
    print(f"{k}: prev={v_old} | curr={v_new}")

print("\n=== Current CT_BASE_CFG (key summary) ===")
for k in ["d_model","n_layers","n_heads","pretrain_epochs","finetune_epochs","pretrain_lr","finetune_lr","finetune_patience","use_amp","mlp_hidden_sizes","mlp_epochs","mlp_lr"]:
    if k in current_base:
        print(f"{k}: {current_base[k]}")

Current: /hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/results/ct_config_ct_v3.5_max_20260207_093208.json
Previous: /hpc2hdd/home/yjiang909/Transfer learning/Corporate_Carbon_Emissions_Transfer_Learning/results/ct_config_backfilled_ct_experiments_stratified_20260205_172919.json

=== C_CT vs current CT_FULL_CFG ===
config_version: prev=<missing> | curr=v3.5_max
device: prev=<missing> | curr=cuda
experiment_code: prev=C_CT | curr=<missing>
finetune_batch_size: prev=<missing> | curr=128
finetune_epochs: prev=<missing> | curr=1200
finetune_lr: prev=2.9999999999999997e-05 | curr=1e-05
finetune_patience: prev=30 | curr=120
finetune_weight_decay: prev=0.0001 | curr=5e-05
freeze_encoder: prev=<missing> | curr=False
loader_num_workers: prev=<missing> | curr=16
loader_persistent_workers: prev=<missing> | curr=True
loader_pin_memory: prev=<missing> | curr=True
loader_prefetch_factor: prev=<missing> | curr=4
lora_alpha: prev=<missing> | curr=128
lora_rank: 